In [11]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.data_processing import (
    load_data,
    create_customer_dataset
)

from src.target_engineering import create_rfm_target

In [4]:
df = load_data("../data/raw/data.csv")

features = create_customer_dataset(df)

target = create_rfm_target(df)

data = features.merge(
    target,
    on="CustomerId"
)

In [5]:
import numpy as np

def calculate_iv(df, feature, target):

    temp = df[[feature, target]].copy()

    temp["bin"] = pd.qcut(
        temp[feature],
        q=10,
        duplicates="drop"
    )

    grouped = temp.groupby("bin")[target].agg(
        good=lambda x: (x == 0).sum(),
        bad=lambda x: (x == 1).sum()
    )

    grouped["good_pct"] = grouped["good"] / grouped["good"].sum()
    grouped["bad_pct"] = grouped["bad"] / grouped["bad"].sum()

    grouped["woe"] = np.log(
        (grouped["good_pct"] + 0.0001) /
        (grouped["bad_pct"] + 0.0001)
    )

    grouped["iv"] = (
        grouped["good_pct"] -
        grouped["bad_pct"]
    ) * grouped["woe"]

    return grouped["iv"].sum()

In [6]:
features_to_test = [
    "transaction_count",
    "total_transaction_amount",
    "avg_transaction_amount",
    "std_transaction_amount"
]

for col in features_to_test:
    print(
        col,
        round(
            calculate_iv(
                data,
                col,
                "is_high_risk"
            ),
            4
        )
    )

transaction_count 0.8728
total_transaction_amount 0.4841
avg_transaction_amount 0.1827
std_transaction_amount 0.478


In [7]:
def calculate_iv(df, feature, target):
    lst = []

    for val in df[feature].unique():
        temp = df[df[feature] == val]
        good = (temp[target] == 0).sum()
        bad = (temp[target] == 1).sum()
        lst.append((val, good, bad))

    iv = 0
    for val, good, bad in lst:
        dist_good = good / df[df[target] == 0].shape[0]
        dist_bad = bad / df[df[target] == 1].shape[0]

        if dist_good == 0 or dist_bad == 0:
            continue

        woe = np.log(dist_good / dist_bad)
        iv += (dist_good - dist_bad) * woe

    return iv

In [8]:
features = [
    "transaction_count",
    "total_transaction_amount",
    "avg_transaction_amount"
]

for col in features:
    print(col, calculate_iv(data, col, "is_high_risk"))

transaction_count 0.6339916418869769
total_transaction_amount 0.730437875064923
avg_transaction_amount 0.7376026083220717


In [9]:
import numpy as np
import pandas as pd

def calculate_iv(df, feature, target, bins=10):

    lst = []

    df = df[[feature, target]].copy()
    df[feature] = pd.qcut(df[feature], q=bins, duplicates="drop")

    grouped = df.groupby(feature)[target].agg(["count", "sum"])
    grouped.columns = ["total", "bad"]

    grouped["good"] = grouped["total"] - grouped["bad"]

    grouped["bad_rate"] = grouped["bad"] / grouped["bad"].sum()
    grouped["good_rate"] = grouped["good"] / grouped["good"].sum()

    grouped["woe"] = np.log(grouped["good_rate"] / grouped["bad_rate"])
    grouped["iv"] = (grouped["good_rate"] - grouped["bad_rate"]) * grouped["woe"]

    return grouped["iv"].sum()

In [14]:
from src.feature_selection import calculate_iv

features = [
    "transaction_count",
    "total_transaction_amount",
    "avg_transaction_amount"
]

for col in features:
    iv = calculate_iv(data, col, "is_high_risk")
    print(col, iv)

transaction_count 0.8740468653259702
total_transaction_amount 0.48465728747490316
avg_transaction_amount 0.18289545675233024
